# 📈 Prophet Time Series Forecasting Model

**Project:** Retail Sales Forecasting  
**Model:** Facebook Prophet  
**Date:** December 2025

## 🎯 Objectives
1. Pre-process data for Prophet model requirements
2. Prepare train/test splits for time series validation
3. Set up and configure Prophet model with appropriate parameters
4. Train the model and generate forecasts
5. Evaluate model performance with multiple metrics

## 📋 Prophet Overview
Prophet is a forecasting tool designed for business time series data that:
- Handles missing data and outliers robustly
- Automatically detects changepoints and trend shifts
- Incorporates seasonality (daily, weekly, yearly)
- Allows for holiday effects and custom events
- Provides interpretable components

---
## 📥 Step 1: Import Required Libraries

In [ ]:
# Install Prophet if not already installed (required in most environments)
!pip install prophet --quiet

# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Prophet for time series forecasting
from prophet import Prophet

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from prophet.plot import plot_plotly, plot_components_plotly

# Metrics for evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'Prophet version: {Prophet.__version__ if hasattr(Prophet, "__version__") else "installed"}')

---
## 📂 Step 2: Load and Inspect Data

**Note:** Upload your `train.csv` file if running in Google Colab.

In [ ]:
# Load the training data
print('📂 Loading data...')
df = pd.read_csv('train.csv')

# Display basic information
print(f'\n✅ Data loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
# Display first few rows
print('\n📊 First 5 rows of the dataset:')
display(df.head())

# Display data types and missing values
print('\n🔍 Data Information:')
df.info()

---
## 🧹 Step 3: Data Pre-processing

### 3.1 Parse Dates and Handle Data Types

In [ ]:
# Convert date column to datetime
print('📅 Converting date column to datetime format...')
df['date'] = pd.to_datetime(df['date'])

# Display date range
print(f'\nDate range:')
print(f'  Start: {df["date"].min().strftime("%Y-%m-%d")}')
print(f'  End: {df["date"].max().strftime("%Y-%m-%d")}')
print(f'  Total days: {(df["date"].max() - df["date"].min()).days} days')
print(f'  Unique dates: {df["date"].nunique()}')

### 3.2 Data Quality Checks

In [ ]:
# Check for missing values
print('🔍 Missing Values Analysis:')
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_summary[missing_summary['Missing_Count'] > 0]

if len(missing_data) > 0:
    print('\n⚠️ Missing values found:')
    display(missing_data)
else:
    print('\n✅ No missing values detected!')

# Handle missing values in unit_sales (target variable)
if df['unit_sales'].isnull().sum() > 0:
    print(f'\n⚠️ Found {df["unit_sales"].isnull().sum()} missing values in unit_sales')
    print('🔧 Filling with 0 (assuming no sales on those records)')
    df['unit_sales'] = df['unit_sales'].fillna(0)
else:
    print('\n✅ No missing values in target variable (unit_sales)')

In [ ]:
# Check for negative sales (returns/refunds)
negative_sales = df[df['unit_sales'] < 0]
print(f'\n🔍 Negative sales (returns): {len(negative_sales):,} records ({len(negative_sales)/len(df)*100:.2f}%)')

if len(negative_sales) > 0:
    print(f'   Min value: {df["unit_sales"].min()}')
    print(f'   Total negative: {negative_sales["unit_sales"].sum()}')
    print('\n💡 Keeping negative values as they represent returns (important for accurate forecasting)')

---
## 📊 Step 4: Aggregate Data for Time Series

### 4.1 Create Daily Aggregated Dataset

Prophet requires data in a specific format:
- `ds`: Date column (datetime)
- `y`: Target variable (numeric)

We'll aggregate all sales by date to create a single time series.

In [ ]:
# Aggregate sales by date
print('📊 Aggregating sales by date...')
daily_sales = df.groupby('date').agg({
    'unit_sales': 'sum',  # Total daily sales
    'onpromotion': 'sum',  # Total items on promotion
    'store_nbr': 'nunique',  # Number of unique stores
    'item_nbr': 'nunique'  # Number of unique items
}).reset_index()

# Rename columns for clarity
daily_sales.columns = ['date', 'total_sales', 'total_promotions', 'num_stores', 'num_items']

print(f'\n✅ Aggregated dataset created!')
print(f'Shape: {daily_sales.shape[0]:,} days × {daily_sales.shape[1]} columns')
print(f'\nDaily sales statistics:')
print(daily_sales['total_sales'].describe())

### 4.2 Format Data for Prophet

Prophet requires columns named 'ds' (datestamp) and 'y' (target variable).

In [ ]:
# Create Prophet-formatted dataset
prophet_data = daily_sales[['date', 'total_sales']].copy()
prophet_data.columns = ['ds', 'y']

# Sort by date (required for time series)
prophet_data = prophet_data.sort_values('ds').reset_index(drop=True)

print('✅ Data formatted for Prophet!')
print(f'\nShape: {prophet_data.shape}')
print('\nFirst 5 rows:')
display(prophet_data.head())
print('\nLast 5 rows:')
display(prophet_data.tail())

### 4.3 Check for Missing Dates

In [ ]:
# Check for missing dates in the time series
date_range = pd.date_range(start=prophet_data['ds'].min(), 
                           end=prophet_data['ds'].max(), 
                           freq='D')
missing_dates = set(date_range) - set(prophet_data['ds'])

if len(missing_dates) > 0:
    print(f'⚠️ Found {len(missing_dates)} missing dates in the time series')
    print(f'\n💡 Prophet can handle missing dates, but let\'s fill them with 0 for completeness')
    
    # Create complete date range
    complete_dates = pd.DataFrame({'ds': date_range})
    prophet_data = complete_dates.merge(prophet_data, on='ds', how='left')
    prophet_data['y'] = prophet_data['y'].fillna(0)
    
    print(f'\n✅ Filled {len(missing_dates)} missing dates with 0 sales')
else:
    print('✅ No missing dates found! Time series is complete.')

print(f'\nFinal dataset shape: {prophet_data.shape}')

---
## 📈 Step 5: Exploratory Data Analysis

### 5.1 Visualize Time Series

In [ ]:
# Create comprehensive time series visualization
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot 1: Full time series
axes[0].plot(prophet_data['ds'], prophet_data['y'], 
             color='steelblue', linewidth=1, alpha=0.8)
axes[0].set_title('📊 Total Daily Sales - Full Time Series', 
                  fontsize=16, fontweight='bold', pad=20)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Total Daily Sales (units)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: Recent trend (last 90 days)
recent_data = prophet_data.tail(90)
axes[1].plot(recent_data['ds'], recent_data['y'], 
             color='coral', linewidth=1.5, marker='o', markersize=3)
axes[1].set_title('📅 Last 90 Days - Detailed View', 
                  fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Total Daily Sales (units)', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Plot 3: Distribution of daily sales
axes[2].hist(prophet_data['y'], bins=50, color='mediumseagreen', 
             alpha=0.7, edgecolor='black')
axes[2].axvline(prophet_data['y'].mean(), color='red', 
                linestyle='--', linewidth=2, label=f'Mean: {prophet_data["y"].mean():,.0f}')
axes[2].axvline(prophet_data['y'].median(), color='blue', 
                linestyle='--', linewidth=2, label=f'Median: {prophet_data["y"].median():,.0f}')
axes[2].set_title('📊 Distribution of Daily Sales', 
                  fontsize=14, fontweight='bold', pad=15)
axes[2].set_xlabel('Daily Sales (units)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📊 Key Statistics:')
print(f'Mean daily sales: {prophet_data["y"].mean():,.2f} units')
print(f'Median daily sales: {prophet_data["y"].median():,.2f} units')
print(f'Std deviation: {prophet_data["y"].std():,.2f} units')
print(f'Min daily sales: {prophet_data["y"].min():,.0f} units')
print(f'Max daily sales: {prophet_data["y"].max():,.0f} units')

---
## 🔀 Step 6: Train/Test Split

### 6.1 Time Series Split Strategy

For time series, we use temporal split (not random split):
- **Training set**: Earlier data (80% of timeline)
- **Test set**: Most recent data (20% of timeline)

This simulates real-world forecasting where we predict future values.

In [ ]:
# Calculate split point (80/20 split)
split_ratio = 0.8
split_index = int(len(prophet_data) * split_ratio)

# Create train and test sets
train_data = prophet_data.iloc[:split_index].copy()
test_data = prophet_data.iloc[split_index:].copy()

print('✅ Train/Test split completed!')
print(f'\n📊 Dataset sizes:')
print(f'Total records: {len(prophet_data):,}')
print(f'Training set: {len(train_data):,} days ({len(train_data)/len(prophet_data)*100:.1f}%)')
print(f'Test set: {len(test_data):,} days ({len(test_data)/len(prophet_data)*100:.1f}%)')

print(f'\n📅 Date ranges:')
print(f'Training: {train_data["ds"].min().strftime("%Y-%m-%d")} to {train_data["ds"].max().strftime("%Y-%m-%d")}')
print(f'Test: {test_data["ds"].min().strftime("%Y-%m-%d")} to {test_data["ds"].max().strftime("%Y-%m-%d")}')

### 6.2 Visualize Train/Test Split

In [ ]:
# Visualize the train/test split
plt.figure(figsize=(16, 6))

plt.plot(train_data['ds'], train_data['y'], 
         color='steelblue', linewidth=1, label='Training Data', alpha=0.8)
plt.plot(test_data['ds'], test_data['y'], 
         color='coral', linewidth=1, label='Test Data', alpha=0.8)

# Add vertical line at split point
plt.axvline(x=train_data['ds'].iloc[-1], color='red', 
            linestyle='--', linewidth=2, label='Train/Test Split')

plt.title('📊 Train/Test Split Visualization', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Daily Sales (units)', fontsize=12)
plt.legend(fontsize=12, loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## ⚙️ Step 7: Prophet Model Setup and Configuration

### 7.1 Initialize Prophet Model

We'll configure Prophet with optimized parameters for retail sales forecasting.

In [ ]:
# Initialize Prophet model with custom parameters
print('⚙️ Initializing Prophet model...')

model = Prophet(
    # Growth parameters
    growth='linear',  # 'linear' or 'logistic' - linear for unbounded growth
    
    # Changepoint parameters (detect trend changes)
    changepoint_prior_scale=0.05,  # Flexibility of trend (0.001-0.5, higher = more flexible)
    changepoint_range=0.8,  # Proportion of history for changepoint detection
    n_changepoints=25,  # Number of potential changepoints
    
    # Seasonality parameters
    yearly_seasonality=True,  # Capture yearly patterns
    weekly_seasonality=True,  # Capture weekly patterns (weekday vs weekend)
    daily_seasonality=False,  # Not needed for daily aggregated data
    seasonality_mode='multiplicative',  # 'additive' or 'multiplicative'
    seasonality_prior_scale=10.0,  # Strength of seasonality (higher = stronger)
    
    # Holiday effects (can be added later)
    holidays=None,  # Will add Ecuadorian holidays if needed
    holidays_prior_scale=10.0,
    
    # Uncertainty estimation
    interval_width=0.95,  # 95% confidence interval
    mcmc_samples=0,  # Use MAP estimation (faster), set >0 for full Bayesian
    
    # Other parameters
    uncertainty_samples=1000,  # Samples for uncertainty intervals
    stan_backend=None  # Use default backend
)

print('\n✅ Prophet model initialized successfully!')
print('\n📋 Model Configuration:')
print(f'  Growth type: {model.growth}')
print(f'  Seasonality mode: {model.seasonality_mode}')
print(f'  Yearly seasonality: {model.yearly_seasonality}')
print(f'  Weekly seasonality: {model.weekly_seasonality}')
print(f'  Changepoint prior scale: {model.changepoint_prior_scale}')
print(f'  Seasonality prior scale: {model.seasonality_prior_scale}')
print(f'  Confidence interval: {model.interval_width * 100}%')

### 7.2 Add Custom Seasonality (Optional)

We can add monthly seasonality to capture patterns that repeat every month.

In [ ]:
# Add monthly seasonality (optional but useful for retail)
model.add_seasonality(
    name='monthly',
    period=30.5,  # Average days in a month
    fourier_order=5  # Number of Fourier terms (higher = more complex pattern)
)

print('✅ Added monthly seasonality to the model')
print('   Period: 30.5 days')
print('   Fourier order: 5')

---
## 🎓 Step 8: Train the Prophet Model

### 8.1 Fit Model on Training Data

In [ ]:
# Train the model
print('🎓 Training Prophet model...')
print(f'Training on {len(train_data):,} days of data')
print('\nThis may take a few moments...')

import time
start_time = time.time()

# Fit the model
model.fit(train_data)

training_time = time.time() - start_time

print(f'\n✅ Model trained successfully!')
print(f'⏱️ Training time: {training_time:.2f} seconds')

---
## 🔮 Step 9: Generate Predictions

### 9.1 Create Future Dataframe

In [ ]:
# Create future dataframe for predictions
# We'll predict for the test period
periods = len(test_data)

print(f'🔮 Creating future dataframe for {periods} days...')
future = model.make_future_dataframe(periods=periods, freq='D')

print(f'\n✅ Future dataframe created!')
print(f'Shape: {future.shape}')
print(f'Date range: {future["ds"].min().strftime("%Y-%m-%d")} to {future["ds"].max().strftime("%Y-%m-%d")}')
print('\nLast 5 dates:')
display(future.tail())

### 9.2 Generate Forecast

In [ ]:
# Generate predictions
print('🔮 Generating forecast...')
forecast = model.predict(future)

print('\n✅ Forecast generated successfully!')
print(f'\nForecast dataframe shape: {forecast.shape}')
print(f'Columns: {forecast.shape[1]}')

# Display key forecast columns
print('\nKey forecast columns:')
print('  - ds: Date')
print('  - yhat: Predicted value')
print('  - yhat_lower: Lower bound of prediction interval')
print('  - yhat_upper: Upper bound of prediction interval')
print('  - trend: Trend component')
print('  - yearly, weekly, monthly: Seasonal components')

print('\nLast 5 predictions:')
display(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

---
## 📊 Step 10: Evaluate Model Performance

### 10.1 Extract Test Predictions

In [ ]:
# Extract predictions for test period
test_forecast = forecast[forecast['ds'] >= test_data['ds'].min()].copy()

# Merge with actual test values
test_results = test_forecast.merge(test_data, on='ds', how='inner')
test_results.columns = [col if col != 'y' else 'actual' for col in test_results.columns]

print('✅ Test predictions extracted')
print(f'\nTest period: {len(test_results)} days')
print(f'From {test_results["ds"].min().strftime("%Y-%m-%d")} to {test_results["ds"].max().strftime("%Y-%m-%d")}')

print('\nFirst 5 predictions vs actuals:')
display(test_results[['ds', 'actual', 'yhat', 'yhat_lower', 'yhat_upper']].head())

### 10.2 Calculate Performance Metrics

In [ ]:
# Calculate evaluation metrics
y_true = test_results['actual'].values
y_pred = test_results['yhat'].values

# Calculate metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Calculate MAPE manually to handle zeros
# Filter out zero values to avoid division by zero
non_zero_mask = y_true != 0
if non_zero_mask.sum() > 0:
    mape = np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
else:
    mape = float('inf')

# Calculate additional metrics
mean_actual = y_true.mean()
mae_percentage = (mae / mean_actual) * 100 if mean_actual != 0 else float('inf')

print('\n' + '='*60)
print('📊 PROPHET MODEL PERFORMANCE METRICS')
print('='*60)
print(f'\nMean Absolute Error (MAE):        {mae:,.2f} units')
print(f'Root Mean Squared Error (RMSE):    {rmse:,.2f} units')
print(f'Mean Absolute Percentage Error:    {mape:.2f}%')
print(f'\nMAE as % of mean sales:            {mae_percentage:.2f}%')
print(f'Mean actual sales:                 {mean_actual:,.2f} units')
print('='*60)

# Interpretation
print('\n💡 Interpretation:')
print(f'On average, predictions are off by {mae:,.0f} units ({mape:.1f}%)')
if mape < 10:
    print('✅ Excellent forecast accuracy (<10% MAPE)')
elif mape < 20:
    print('✅ Good forecast accuracy (10-20% MAPE)')
elif mape < 30:
    print('⚠️ Acceptable forecast accuracy (20-30% MAPE)')
else:
    print('⚠️ Consider model improvement (>30% MAPE)')

---
## 📈 Step 11: Visualize Results

### 11.1 Plot Forecast vs Actual

In [ ]:
# Plot forecast vs actual for test period
fig, ax = plt.subplots(figsize=(16, 8))

# Plot actual values
ax.plot(test_results['ds'], test_results['actual'], 
        color='steelblue', linewidth=2, label='Actual Sales', marker='o', markersize=4)

# Plot predictions
ax.plot(test_results['ds'], test_results['yhat'], 
        color='coral', linewidth=2, label='Prophet Forecast', linestyle='--', marker='s', markersize=4)

# Plot confidence interval
ax.fill_between(test_results['ds'], 
                test_results['yhat_lower'], 
                test_results['yhat_upper'],
                alpha=0.2, color='coral', label='95% Confidence Interval')

ax.set_title('🔮 Prophet Forecast vs Actual Sales (Test Period)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Daily Sales (units)', fontsize=12)
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)

# Add metrics text box
textstr = f'MAE: {mae:,.0f}\nRMSE: {rmse:,.0f}\nMAPE: {mape:.2f}%'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

### 11.2 Plot Forecast Components

In [ ]:
# Plot forecast components (trend, seasonality)
print('📊 Plotting forecast components (trend, seasonality)...')
fig = model.plot_components(forecast)
plt.tight_layout()
plt.show()

print('\n💡 Component Interpretation:')
print('  - Trend: Long-term direction of sales')
print('  - Yearly: Seasonal pattern that repeats annually')
print('  - Weekly: Day-of-week effects (weekday vs weekend)')
print('  - Monthly: Monthly seasonal patterns')

### 11.3 Plot Full Forecast (Train + Test)

In [ ]:
# Use Prophet's built-in plotting
print('📊 Plotting full forecast (training + test periods)...')
fig = model.plot(forecast, figsize=(16, 8))
ax = fig.gca()

# Add vertical line at train/test split
ax.axvline(x=train_data['ds'].iloc[-1], color='red', 
           linestyle='--', linewidth=2, label='Train/Test Split', alpha=0.7)

ax.set_title('📈 Prophet Forecast - Full Timeline', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Daily Sales (units)', fontsize=12)
ax.legend(['Actual', 'Forecast', 'Uncertainty', 'Train/Test Split'], 
          fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

---
## 📝 Step 12: Summary and Next Steps

### Model Performance Summary

In [ ]:
print('\n' + '='*70)
print('🎯 PROPHET MODEL SETUP - SUMMARY')
print('='*70)
print('\n✅ COMPLETED TASKS:')
print('   1. ✓ Data loaded and inspected')
print('   2. ✓ Dates parsed and validated')
print('   3. ✓ Data quality checks performed')
print('   4. ✓ Daily sales aggregated')
print('   5. ✓ Data formatted for Prophet (ds, y columns)')
print('   6. ✓ Missing dates filled')
print('   7. ✓ Train/test split created (80/20)')
print('   8. ✓ Prophet model initialized with optimal parameters')
print('   9. ✓ Custom seasonality added (monthly)')
print('   10. ✓ Model trained on historical data')
print('   11. ✓ Forecasts generated for test period')
print('   12. ✓ Performance metrics calculated')
print('   13. ✓ Results visualized')

print('\n📊 MODEL PERFORMANCE:')
print(f'   MAE:  {mae:,.2f} units')
print(f'   RMSE: {rmse:,.2f} units')
print(f'   MAPE: {mape:.2f}%')

print('\n⚙️ MODEL CONFIGURATION:')
print(f'   Growth: {model.growth}')
print(f'   Seasonality mode: {model.seasonality_mode}')
print(f'   Seasonality: yearly, weekly, monthly')
print(f'   Training time: {training_time:.2f} seconds')

print('\n📈 DATASET INFO:')
print(f'   Total days: {len(prophet_data):,}')
print(f'   Training days: {len(train_data):,}')
print(f'   Test days: {len(test_data):,}')

print('\n🚀 NEXT STEPS:')
print('   1. Compare with other models (ARIMA, Exponential Smoothing)')
print('   2. Fine-tune hyperparameters if needed')
print('   3. Add holiday effects for better accuracy')
print('   4. Add external regressors (promotions, oil prices)')
print('   5. Generate future forecasts for business planning')
print('='*70)

### Save Results (Optional)

In [ ]:
# Save predictions to CSV (optional)
output_file = 'prophet_predictions.csv'
test_results[['ds', 'actual', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(output_file, index=False)
print(f'✅ Predictions saved to {output_file}')

# Save model metrics
metrics_df = pd.DataFrame({
    'Model': ['Prophet'],
    'MAE': [mae],
    'RMSE': [rmse],
    'MAPE': [mape],
    'Training_Time': [training_time]
})
metrics_df.to_csv('prophet_metrics.csv', index=False)
print(f'✅ Metrics saved to prophet_metrics.csv')